# How big are New York's affordable housing buildings?

New York City's housing department (HPD) publishes data on buildings involved in the city's affordable housing plans since 2014. I downloaded the [Affordable Housing Production by Building dataset](https://data.cityofnewyork.us/d/hg8x-zxpr) from NYC Open Data.

I wanted to find out how many apartments a typical building has. I'll calculate the mean, median and mode, first with pandas and then by hand. I'll also look at how common very large buildings actually are.

I'm using the `Total Units` column, which includes affordable and market-rate apartments. The data is organized by building-project record, so the same building can appear more than once.

And since we live in Mamdani's New York, I couldn't resist bringing housing politics into it.

In [1]:
import pandas as pd

## Loading the data

I initially ran into a small problem with the CSV. A few buildings had their unit counts written with a comma, like `1,175`. Pandas treated the column as text, which meant I couldn't calculate the average.

The fix was simple: `thousands=","` tells pandas to read those values as numbers.

I'll also check how many rows there are and whether any unit counts are missing before doing the calculations.

In [2]:
housing = pd.read_csv("affordable_housing.csv", thousands=",")

housing[["Borough", "Total Units"]].head()

,Borough,Total Units
0,Queens,43
1,Brooklyn,187
2,Manhattan,25
3,Manhattan,16
4,Manhattan,15


In [3]:
print(f"Rows: {len(housing):,}")
print(f"Columns: {len(housing.columns)}")
print(f"Missing unit counts: {housing['Total Units'].isna().sum()}")
print(f"Smallest unit count: {housing['Total Units'].min():,}")
print(f"Largest unit count: {housing['Total Units'].max():,}")

assert 1_000 <= len(housing) <= 1_000_000

Rows: 9,551
Columns: 41
Missing unit counts: 0
Smallest unit count: 1
Largest unit count: 1,320


## Mean, median and mode

There are three ways I can describe a typical building in this dataset.

The mean is the average number of units per record. The median is the middle value once the records are sorted by size. The mode is the number of units that appears most often.

I'm curious whether these three numbers will be similar or whether a few very large buildings make a difference.

I'll start with pandas, which can calculate all three directly.

In [4]:
units = housing["Total Units"].dropna()

pandas_mean = units.mean()
pandas_median = units.median()
pandas_modes = sorted(units.mode().tolist())

print(f"Mean: {pandas_mean:.2f} units")
print(f"Median: {pandas_median:.0f} units")
print(f"Mode(s): {pandas_modes}")

Mean: 46.65 units
Median: 12 units
Mode(s): [1]


## The hard way: no pandas

Now I'll repeat the same three calculations without pandas or the `statistics` module.

First, I'll read the CSV again using Python's built-in `csv` module and collect all the unit counts in a regular list. This part is completely separate from the pandas DataFrame.

I'll also need to remove commas from large numbers before converting them into integers.

Then I'll write my own functions for the mean, median and mode.

In [5]:
import csv

unit_counts = []

with open(
    "affordable_housing.csv",
    newline="",
    encoding="utf-8-sig",
) as file:
    reader = csv.DictReader(file)

    for row in reader:
        raw_units = row["Total Units"]

        if raw_units is None:
            continue

        units_text = raw_units.strip().replace(",", "")

        if not units_text:
            continue

        try:
            unit_counts.append(int(units_text))
        except ValueError:
            continue

print(f"Number of unit counts: {len(unit_counts):,}")
print(f"First five values: {unit_counts[:5]}")

Number of unit counts: 9,551
First five values: [43, 187, 25, 16, 15]


### Mean

For the mean, I'll add up all the unit counts using a loop, then divide the total by the number of records.

In [6]:
def calculate_mean(numbers):
    """Mean of a list of numbers."""
    total = 0

    for number in numbers:
        total += number

    return total / len(numbers)


manual_mean = calculate_mean(unit_counts)
print(f"Manual mean: {manual_mean:.2f} units")

Manual mean: 46.65 units


### Median

The median is the value in the middle once all records are sorted by size.

With an odd number of records, I take the middle value. With an even number, there is no single middle, so I take the average of the two values around the center.

In [7]:
def calculate_median(numbers):
    """Middle value of a list of numbers."""
    ordered = sorted(numbers)
    middle = len(ordered) // 2

    if len(ordered) % 2 == 1:
        return ordered[middle]

    return (ordered[middle - 1] + ordered[middle]) / 2


manual_median = calculate_median(unit_counts)
print(f"Manual median: {manual_median:.0f} units")

Manual median: 12 units


### Mode

The mode is a little different. I need to count how often each number appears.

I'll use a dictionary to keep track of the frequencies, then find the number that occurs most often.

There could also be more than one mode if several values have the same highest frequency.

In [8]:
def calculate_modes(numbers):
    """Return the most frequent values in a list."""
    counts = {}

    for number in numbers:
        if number in counts:
            counts[number] += 1
        else:
            counts[number] = 1

    highest_count = max(counts.values())
    modes = []

    for number in counts:
        if counts[number] == highest_count:
            modes.append(number)

    return sorted(modes)


manual_modes = calculate_modes(unit_counts)
print(f"Manual mode(s): {manual_modes}")

Manual mode(s): [1]


## Checking the results

Now I've calculated the mean, median and mode twice.

I'll check whether pandas and my own functions give the same results. I'll also make sure both methods used exactly the same unit counts.

In [9]:
assert sorted(unit_counts) == sorted(units.tolist())
assert abs(manual_mean - pandas_mean) < 1e-9
assert manual_median == pandas_median
assert manual_modes == pandas_modes

print("Success! All three results match.")

Success! All three results match.


## What the three numbers say

The first thing I noticed is how different the three numbers are. The mean is much higher than the median, while the most common value is just one unit.

I found that pretty surprising. When I think about affordable housing projects in New York, I usually picture large apartment buildings.

Then I came across this meme about a parking lot being turned into more than 200 affordable homes.

![Meme about converting a NYC parking lot into housing, with a sarcastic comment about data centers](housing_meme.jpg)

*Source: [Instagram](https://www.instagram.com/p/DcRoS7ASukK/). The sarcastic comment in the screenshot is attributed to @lukepbeasley.*

The data-center joke made me laugh, but it also gave me an idea. How common are buildings with 200 or more units in the city's housing data?

## How common are large housing projects?

At first, I tried adding up all the units in each borough. But that didn't really answer my question about how big a typical building is.

So I changed the chart.

I'll group the records into five categories based on their total number of units, from single-unit properties to records with 200 or more units.

Then I'll count the records in each category and draw a simple bar chart using Python's built-in string operations.

In [10]:
size_groups = {
    "1 unit": 0,
    "2-10 units": 0,
    "11-50 units": 0,
    "51-199 units": 0,
    "200+ units": 0,
}

for count in unit_counts:
    if count == 1:
        size_groups["1 unit"] += 1
    elif count <= 10:
        size_groups["2-10 units"] += 1
    elif count <= 50:
        size_groups["11-50 units"] += 1
    elif count < 200:
        size_groups["51-199 units"] += 1
    else:
        size_groups["200+ units"] += 1

assert sum(size_groups.values()) == len(unit_counts)

print(size_groups)

{'1 unit': 1823, '2-10 units': 2748, '11-50 units': 2785, '51-199 units': 1651, '200+ units': 544}


In [11]:
largest_group = max(size_groups.values())
bar_width = 24

print("NYC HOUSING RECORDS BY NUMBER OF UNITS")
print("Unit-count range | Number of records")
print("-" * 55)

for label, count in size_groups.items():
    bar_length = round(count / largest_group * bar_width)
    bar = "█" * bar_length

    print(f"{label:<12} | {bar:<24} | {count:>5,}")

large_count = size_groups["200+ units"]
large_share = large_count / len(unit_counts) * 100

print(
    f"\n{large_count:,} records have 200 or more units "
    f"({large_share:.1f}% of all records)."
)

NYC HOUSING RECORDS BY NUMBER OF UNITS
Unit-count range | Number of records
-------------------------------------------------------
1 unit       | ████████████████         | 1,823
2-10 units   | ████████████████████████ | 2,748
11-50 units  | ████████████████████████ | 2,785
51-199 units | ██████████████           | 1,651
200+ units   | █████                    |   544

544 records have 200 or more units (5.7% of all records).


## So what did I find?

Only a small share of the records have 200 or more units. Most are much smaller, which helps explain why the mean is so much higher than the median.

The one-unit mode is also interesting. The dataset includes housing preservation projects, not just newly constructed apartment buildings, so it contains smaller properties as well.

The meme shows the kind of large development that attracts attention online. But those large records make up a relatively small part of this dataset.

One important limitation is that I've counted total units, including market-rate apartments. These numbers tell us something about building size, but not directly about affordability.

## Conclusion

I started by wondering how big New York's affordable housing buildings are. It turned out that the answer depends quite a bit on whether I look at the mean, median or mode.

I also learned more about what the HPD dataset actually contains. Some records are for preservation projects, and the same building can appear more than once. That makes it important to be careful about what these numbers represent.

What I still don't know is how affordable the apartments actually are. For that, I'd need information about rents and household incomes.

Maybe that's a question for my next project. For now, calculating housing statistics in New York has been easier than finding affordable housing.

## Sources

- [NYC HPD: Affordable Housing Production by Building](https://data.cityofnewyork.us/d/hg8x-zxpr)
- [Instagram post of the housing meme](https://www.instagram.com/p/DcRoS7ASukK/)
- Columbia University, School of International Public Affairs (SIPA), Computing in Context (CiC) course materials